In [2]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [3]:
from src_oop.jobs.autopilot.config import unit_gs, autopilot_gs
from src_oop.core.my_gspread import GoogleTabs
import pandas as pd

In [ ]:
class Autopilot:
    def __init__(self):
        # Сохраняем только конфигурацию, не создавая подключений сразу
        self._table_from_name = unit_gs.get("title")
        self._sheet_from_name = unit_gs.get("unit_sheet")
        
        self._table_to_name = autopilot_gs.get("title")
        self._sheet_to_name = autopilot_gs.get("ic_sheet")
        
        # Здесь будем хранить сами объекты соединений после их активации
        self._google_connect_from = None
        self._google_connect_to = None

    @property
    def google_connect_from(self):
        """Ленивое подключение к исходной таблице"""
        if self._google_connect_from is None:
            print(f"Инициализация подключения к {self._table_from_name}")
            self._google_connect_from = GoogleTabs(self._table_from_name, self._sheet_from_name)
        return self._google_connect_from

    @property
    def google_connect_to(self):
        """Ленивое подключение к целевой таблице"""
        if self._google_connect_to is None:
            print(f"Инициализация подключения к {self._table_to_name}")
            self._google_connect_to = GoogleTabs(self._table_to_name, self._sheet_to_name)
        return self._google_connect_to

    def get_unit_data(self):
        # Теперь обращение к self.google_connect_from вызовет создание подключения
        unit_table = self.google_connect_from 
        unit_data = unit_table.sheet_title.get_all_values()
        
        headers = unit_data[0]
        rows = unit_data[1:]
        df = pd.DataFrame(rows, columns=headers)
        
        # Приводим типы
        df['Артикул'] = pd.to_numeric(df['Артикул'], errors='coerce')
        
        df_short = df[['Артикул', 'Цена для клиента', 'Мар', 'ФБС']]
        return df_short

In [5]:
autopilot = Autopilot()
df_unit = autopilot.get_unit_data()

Инициализация подключения к UNIT 2.0 (tested)
✅ Успешное подключение к UNIT 2.0 (tested) -> MAIN (tested)


In [6]:
autopilot_table = autopilot.google_connect_to
autopilot_table.set_df_to_google(df_unit)

Инициализация подключения к Панель управления продажами Вектор...
✅ Успешное подключение к Панель управления продажами Вектор -> ИУ_ИНФО
✅ Успешное подключение к Панель управления продажами Вектор -> ИУ_ИНФО
Таблица полностью обновлена
